# Machine Learning Coursework: Telco Customer Churn Classification
**Individual Assignment**
Binary classification to predict customer churn using KNN, Logistic Regression, and Random Forest.

## 1. Setup & Imports

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

os.makedirs('figures', exist_ok=True)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'font.size': 11, 'figure.autolayout': True})

## 2. Load Dataset & Task 1: Exploratory Data Analysis (EDA)

In [ ]:
df = pd.read_csv('Telco-Customer-Churn.csv')
print(f'Shape: {df.shape}')
print(df['Churn'].value_counts(normalize=True))

# Fix whitespace in TotalCharges
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0.0)

In [ ]:
# Plot Churn Distribution
plt.figure(figsize=(6, 4))
sns.countplot(x='Churn', hue='Churn', data=df, palette=['#4C72B0', '#C44E52'], legend=False)
plt.title('Target Class Distribution (Customer Churn)')
plt.show()

## 3. Task 2: Data Preprocessing

In [ ]:
# Drop ID and map target
df_clean = df.drop(columns=['customerID'])
df_clean['Churn'] = df_clean['Churn'].map({'No': 0, 'Yes': 1})

# One-hot encode categoricals
df_encoded = pd.get_dummies(df_clean, drop_first=True, dtype=int)

X = df_encoded.drop(columns=['Churn'])
y = df_encoded['Churn']

# Stratified Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f'Train shape: {X_train.shape}, Test shape: {X_test.shape}')

## 4. Task 2: Model Training (Setting 1: Baseline vs Setting 2: Tuned)

In [ ]:
results = []

def record(name, setting, y_true, y_pred, y_prob):
    results.append({
        'Model': name,
        'Setting': setting,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1-Score': f1_score(y_true, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_true, y_prob)
    })

# Baseline KNN
knn = KNeighborsClassifier().fit(X_train_scaled, y_train)
record('KNN', 'Baseline', y_test, knn.predict(X_test_scaled), knn.predict_proba(X_test_scaled)[:, 1])

# Baseline Logistic Regression
lr = LogisticRegression(random_state=42, max_iter=1000).fit(X_train_scaled, y_train)
record('Logistic Regression', 'Baseline', y_test, lr.predict(X_test_scaled), lr.predict_proba(X_test_scaled)[:, 1])

# Baseline Random Forest
rf = RandomForestClassifier(random_state=42).fit(X_train, y_train)
record('Random Forest', 'Baseline', y_test, rf.predict(X_test), rf.predict_proba(X_test)[:, 1])

In [ ]:
# Tuned Models via GridSearchCV
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Tuned KNN
grid_knn = GridSearchCV(KNeighborsClassifier(), {'n_neighbors': [5, 11, 21], 'metric': ['euclidean', 'manhattan']}, cv=cv, scoring='roc_auc', n_jobs=-1).fit(X_train_scaled, y_train)
best_knn = grid_knn.best_estimator_
record('KNN', 'Tuned', y_test, best_knn.predict(X_test_scaled), best_knn.predict_proba(X_test_scaled)[:, 1])

# Tuned LR
grid_lr = GridSearchCV(LogisticRegression(random_state=42, max_iter=1000), {'C': [0.1, 1.0, 10.0], 'class_weight': [None, 'balanced']}, cv=cv, scoring='roc_auc', n_jobs=-1).fit(X_train_scaled, y_train)
best_lr = grid_lr.best_estimator_
record('Logistic Regression', 'Tuned', y_test, best_lr.predict(X_test_scaled), best_lr.predict_proba(X_test_scaled)[:, 1])

# Tuned RF
grid_rf = GridSearchCV(RandomForestClassifier(random_state=42), {'n_estimators': [100, 200], 'max_depth': [6, 10], 'class_weight': ['balanced']}, cv=cv, scoring='roc_auc', n_jobs=-1).fit(X_train, y_train)
best_rf = grid_rf.best_estimator_
record('Random Forest', 'Tuned', y_test, best_rf.predict(X_test), best_rf.predict_proba(X_test)[:, 1])

# Comparison DataFrame
df_res = pd.DataFrame(results)
display(df_res)

## 5. Task 3: AI Ethics Audit & Post-Deployment Analysis

In [ ]:
# Feature Importance
importances = pd.Series(best_rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)
plt.figure(figsize=(8, 4))
sns.barplot(x=importances.values, y=importances.index, palette='viridis')
plt.title('Top 10 Feature Importances (Random Forest Tuned)')
plt.xlabel('Importance')
plt.show()